## Import scGPT and dependencies

In [3]:
from pathlib import Path
import numpy as np
from scipy.stats import mode
import scanpy as sc
import warnings 
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix
import seaborn as sns 
import pandas as pd
import sys  

In [4]:
!pip install faiss-cpu

In [5]:
!pip install scanpy 

In [6]:
# extra dependency for similarity search
try:
    import faiss 

    faiss_imported = True
except ImportError:
    faiss_imported = False
    print(
        "faiss not installed! We highly recommend installing it for fast similarity search."
    )
    print("To install it, see https://github.com/facebookresearch/faiss/wiki/Installing-Faiss")

warnings.filterwarnings("ignore", category=ResourceWarning)

In [7]:
import os

if not hasattr(os, "sched_getaffinity"):
    os.sched_getaffinity = lambda pid: set(range(os.cpu_count()))

In [8]:
import scgpt as scg     

/opt/miniconda3/envs/scgpt/lib/python3.9/site-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/opt/miniconda3/envs/scgpt/lib/python3.9/site-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/opt/miniconda3/envs/scgpt/lib/python3.9/site-packages/torchtext/vocab/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/opt/miniconda3/envs/scgpt/lib/python3.9/site-packages/torchtext/utils.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this

In [9]:
model_dir = Path("model/")
adata = sc.read_h5ad('data/sample_proc_lung_train.h5ad')
test_adata = sc.read_h5ad('data/sample_proc_lung_test.h5ad') 



In [10]:
adata.obs


,sample,source,cell_type,subclone,complexity
cell_name,,,,,
AAACCTGAGACGACGT_LUNG_T34,P0034,tLung,B_cell,0,1287
AAACCTGAGACTGTAA_LUNG_T34,P0034,tLung,Malignant,1,5804
AAACCTGAGCCAGGAT_EBUS_28,P1028,tL/B,Malignant,1,1728
AAACCTGAGCTAAACA_LUNG_T34,P0034,tLung,Malignant,1,2530
AAACCTGAGCTAGGCA_EBUS_28,P1028,tL/B,Malignant,1,3414
...,...,...,...,...,...
TTTGTCATCCTGCAGG_LUNG_T34,P0034,tLung,Macrophage,0,2227
TTTGTCATCCTTTCTC_LUNG_T06,P0006,tLung,T_cell,0,1869
TTTGTCATCGACGGAA_LUNG_T25,P0025,tLung,B_cell,0,1911


In [11]:
test_adata.obs

,sample,source,cell_type,subclone,complexity
cell_name,,,,,
AAACCTGAGATGTCGG_LUNG_T31,P0031,tLung,Epithelial,0,1162
AAACCTGAGCGCTTAT_LUNG_T19,P0019,tLung,Macrophage,0,1509
AAACCTGAGGCCCTTG_EBUS_49,P1049,tL/B,T_cell,0,1534
AAACCTGAGGTAAACT_EBUS_49,P1049,tL/B,NK_cell,0,1233
AAACCTGCAACTGCGC_LUNG_T31,P0031,tLung,Epithelial,0,1741
...,...,...,...,...,...
TTTGTCATCAGCATGT_LUNG_T19,P0019,tLung,Macrophage,0,2345
TTTGTCATCCACGTGG_EBUS_06,P1006,tL/B,Malignant,1,1275
TTTGTCATCCTCGCAT_LUNG_T19,P0019,tLung,B_cell,0,1513


In [12]:
cell_type_key = "celltype"
gene_col = "gene_name"

## 2. Embed the reference dataset


In [ ]:
ref_embed_adata = scg.tasks.embed_data(
    adata,
    model_dir,
    gene_col=gene_col,
    batch_size=64,
)

scGPT - INFO - match 2978/3000 genes in vocabulary of size 60697.


/opt/miniconda3/envs/scgpt/lib/python3.9/site-packages/scgpt/model/model.py:77: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(
/opt/miniconda3/envs/scgpt/lib/python3.9/site-packages/torch/amp/autocast_mode.py:250: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
Embedding cells:   0%|          | 0/363 [00:00<?, ?it/s]/opt/miniconda3/envs/scgpt/lib/python3.9/site-packages/torch/nn/modules/transformer.py:408: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:180.)
  output = torch._nested_tensor_from_mask(output, src_key_padding_mask.logical_not(), mask_check=False)
Embedding cells:  15%|█▌        | 55/363 [12:51<1:21:58, 15.97s/it]

## 3.Embed the query dataset

In [ ]:
test_embed_adata = scg.tasks.embed_data(
    test_adata,
    model_dir,
    gene_col=gene_col,
    batch_size=64,
)

## 4. Concatenate the two datasets

In [ ]:
adata_concat = test_embed_adata.concatenate(ref_embed_adata, batch_key="dataset")
# mark the reference vs. query dataset
adata_concat.obs["is_ref"] = ["Query"] * len(test_embed_adata) + ["Reference"] * len(
    ref_embed_adata
)
adata_concat.obs["is_ref"] = adata_concat.obs["is_ref"].astype("category")

## 5. Mask the query dataset cell types

In [ ]:
adata_concat.obs[cell_type_key] = adata_concat.obs[cell_type_key].cat.add_categories(["To be predicted"])
adata_concat.obs[cell_type_key][: len(test_embed_adata)] = "To be predicted"

## 6. Visualize the embeddings

In [ ]:
sc.pp.neighbors(adata_concat, use_rep="X_scGPT")
sc.tl.umap(adata_concat)
sc.pl.umap(
    adata_concat, color=["is_ref", cell_type_key], wspace=0.4, frameon=False, ncols=1
)


## 7. Reference mapping and transfer the annotations

In [ ]:
# Those functions are only used when faiss is not installed
def l2_sim(a, b):
    sims = -np.linalg.norm(a - b, axis=1)
    return sims

def get_similar_vectors(vector, ref, top_k=10):
        # sims = cos_sim(vector, ref)
        sims = l2_sim(vector, ref)
        
        top_k_idx = np.argsort(sims)[::-1][:top_k]
        return top_k_idx, sims[top_k_idx]

In [ ]:
ref_cell_embeddings = ref_embed_adata.obsm["X_scGPT"]
test_emebd = test_embed_adata.obsm["X_scGPT"]

k = 10  # number of neighbors

index = faiss.IndexFlatL2(ref_cell_embeddings.shape[1])
index.add(ref_cell_embeddings)

# Query dataset, k - number of closest elements (returns 2 numpy arrays)
distances, labels = index.search(test_emebd, k)

idx_list=[i for i in range(test_emebd.shape[0])]
preds = []
sim_list = distances
for k in idx_list:
    if faiss_imported:
        idx = labels[k]
    else:
        idx, sim = get_similar_vectors(test_emebd[k][np.newaxis, ...], ref_cell_embeddings, k)
    pred = ref_embed_adata.obs[cell_type_key][idx].value_counts()
    preds.append(pred.index[0])
gt = test_adata.obs[cell_type_key].to_numpy()

## 8. Evaluation of Performance

In [ ]:
res_dict = {
    "accuracy": accuracy_score(gt, preds),
    "precision": precision_score(gt, preds, average="macro"),
    "recall": recall_score(gt, preds, average="macro"),
    "macro_f1": f1_score(gt, preds, average="macro"),
}

res_dict

In [ ]:
y_true = gt
y_pred = preds
cell_type_list = np.unique(y_true)
matrix = confusion_matrix(y_true, y_pred, labels=cell_type_list)
matrix = matrix.astype("float") / matrix.sum(axis=1)[:, np.newaxis]

df = pd.DataFrame(matrix, index=cell_type_list[:matrix.shape[0]], columns=cell_type_list[:matrix.shape[1]])

ax = sns.clustermap(df,  
                    cmap='Purples',
                    annot=True ,
                    fmt=".2f", 
                    annot_kws={'size': 16}, 
                    vmin=0, 
                    vmax=1,
                    row_cluster=False, 
                    col_cluster=False, 
                    figsize=(14, 14))